# pPb data versus NLOJet: EPPS21 × CT18 and CT18

Compare combined reconstructed pPb dijet ηCM distributions from notebook 06 to the supplied central spectra in `extended-comparison.root`, separately for each exact pTave interval. Each model has an overlay and a Data/prediction ratio figure. Data use black squares; NLOJet predictions use colored markers.

The observable is a **normalized shape**, not an absolute cross section or RpPb. EPPS21×CT18 uses `npdf_central`; CT18 uses the **pp reference** `pdf_central`, at the supplied 8.16 TeV. Default campaign: the pooled Nov. 09 + partial Nov. 18 sample (two Nov. 18 jobs omitted). Do not substitute campaign/pool ratios or `ratio_central`.

Selections follow the existing combined jetId data workflow and the indexed per-jet |ηCM| ≤ 1.9 acceptance. Exact raw half-open pTave projections are used: MB below 80 GeV, Jet60 for 80–90 and 90–120 GeV (the established nPDF convention, explicitly approved for this comparison), Jet100 for 120–160 and 200–300 GeV. The optional 50–70 interval uses MB. No averaging of normalized narrow bins or mixing triggers is performed.

Data eta bins are summed onto whole common theory edges with variances added. Whole partially accepted boundary bins are retained; exterior theory NaNs are excluded and interior NaNs fail. Flow bins must carry no yield/error. Each spectrum is independently normalized over the same accepted domain; `integral` gives fractions per bin, `bin_width` gives unit-area density. Uniform theory source-width factors cancel. Fixed normalization rescales diagonal errors only, following notebook 06; normalization-induced covariance is not estimated. Independent Data/theory ratio errors include data statistics and stored pre-fit integration statistics. Zero denominators are omitted in display copies. No experimental systematic, PDF, scale or fit-residual uncertainties are included. This comparison uses reconstructed data without unfolding or a nonperturbative correction to NLOJet; differences can include detector and hadronization effects.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'CMakeLists.txt').exists())
sys.path.insert(0, str(PROJECT_ROOT))
from hist_analysis.python.notebook_setup import load_root
ROOT = load_root(batch=True)
ROOT.gStyle.SetOptStat(0)
ROOT.TH1.AddDirectory(False)
from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import NPDF_TRIGGER_BINS
from hist_analysis.python.histogram_io import resolve_data_file
from hist_analysis.python.extended_theory_comparison import prepare_comparison, MODELS
from hist_analysis.python.data_mc_plotting import draw_comparison_figure, selection_lines
from hist_analysis.python.root_style import configure_cms_header, configure_mplhep_export
from hist_analysis.python.plot_labels import CMS_PRELIMINARY_HEADER, PPB_HEADER_RIGHT
from hist_analysis.python.output_naming import OutputDirectory, OutputSelection
from hist_analysis.python.notebook_output import NotebookRootOutput
from IPython.display import display

In [ ]:
THEORY_FILE = Path(os.environ.get('EXTENDED_THEORY_FILE', PROJECT_ROOT/'hist_analysis/root_files/extended-comparison.root'))
DATA_DIR = Path(os.environ.get('PPB_DATA_DIR', BASE_DIR/'exp'))
DATA_SELECTION = 'jetId'
DATA_DIRECTION = 'combined'
ETA_CUT_VALUE = 1.9
CAMPAIGN = 'combined'  # combined, nov09, nov18
ETA_NORMALIZATION = 'integral'  # or bin_width
PTAVE_BINS = {trigger: list(bins) for trigger, bins in NPDF_TRIGGER_BINS.items()}
# Optional exact stored bin: PTAVE_BINS['mb'].append((50, 70))
CMS_HEADER_LEFT = CMS_PRELIMINARY_HEADER
CMS_HEADER_RIGHT = PPB_HEADER_RIGHT
SAVE_MPLHEP = True
SAVE_PNG = False
DRAW_GRID = True
OVERLAY_Y_RANGE = (0., 0.15)
RATIO_Y_RANGE = (0.75, 1.25)
CURVE_STYLES = {'Data': (ROOT.kBlack, 21), 'EPPS21 x CT18': (ROOT.kRed+1, 20),
                'CT18 (pp reference)': (ROOT.kBlue+1, 24)}
configure_cms_header(CMS_HEADER_LEFT, CMS_HEADER_RIGHT)
configure_mplhep_export(SAVE_MPLHEP, save_png=SAVE_PNG)
DATA_FILES = {t: resolve_data_file(DATA_DIR, t, DATA_DIRECTION, DATA_SELECTION) for t in PTAVE_BINS}
OUTPUT_SELECTIONS = OutputSelection(DATA_SELECTION, ETA_CUT_VALUE, PTAVE_BINS)
OUTPUT_DIR = OutputDirectory(PROJECT_ROOT/'hist_analysis/output/dijet_data_vs_extended_theory', OUTPUT_SELECTIONS)
ROOT_OUTPUT = NotebookRootOutput(OUTPUT_DIR, '07_dijet_data_vs_extended_theory')
INPUT_PROVENANCE = {'theory': str(THEORY_FILE.resolve()), 'sha256': hashlib.sha256(THEORY_FILE.read_bytes()).hexdigest(),
                    'data': {t: str(p.resolve()) for t,p in DATA_FILES.items()}}
print(INPUT_PROVENANCE)

In [ ]:
RESULTS = {}
CANVASES = {}
for trigger, intervals in PTAVE_BINS.items():
    for interval in intervals:
        result = prepare_comparison(DATA_FILES[trigger], THEORY_FILE, interval,
            eta_cut=ETA_CUT_VALUE, campaign=CAMPAIGN, normalization=ETA_NORMALIZATION)
        RESULTS[(trigger, interval)] = result
        annotations = (*selection_lines(PROJECT_ROOT, interval, eta_cut=ETA_CUT_VALUE,
            direction=DATA_DIRECTION, selection=DATA_SELECTION), 'Shapes; statistical errors only')
        for index, model in enumerate(MODELS):
            tag = f'{CAMPAIGN}_model{index}_ptave_{interval[0]}_{interval[1]}_{ETA_NORMALIZATION}'
            for kind in ('overlay', 'ratio'):
                histograms = {model: result['shapes'][model], 'Data': result['shapes']['Data']} if kind == 'overlay' else {f'Data / {model}': result['ratios'][model]}
                styles = CURVE_STYLES if kind == 'overlay' else {f'Data / {model}': CURVE_STYLES['Data']}
                canvas = draw_comparison_figure(histograms, name=f'c_{tag}_{kind}',
                    x_title='#eta_{CM}^{dijet}',
                    y_title=('dN/d#eta_{CM}^{dijet}' if ETA_NORMALIZATION == 'integral' else '1/N dN/d#eta_{CM}^{dijet}') if kind == 'overlay' else f'Data / {model}',
                    x_range=result['x_range'], styles=styles, annotations=annotations,
                    y_range=OVERLAY_Y_RANGE if kind == 'overlay' else RATIO_Y_RANGE,
                    reference=kind == 'ratio', grid=DRAW_GRID,
                    output=OUTPUT_DIR/f'{tag}_{kind}.pdf', save_png=SAVE_PNG)
                CANVASES[(trigger, interval, model, kind)] = canvas
                display(canvas)

## Numerical validation and archive
Check unit normalization and every valid ratio against the explicit independent-error formula. Further regression tests compare raw data projections with direct TH2 bin sums and theory values with the stored source bins. These tests verify arithmetic and provenance, not detector-level agreement with perturbative theory.

In [ ]:
import math
VALIDATION = []
for (trigger, interval), result in RESULTS.items():
    for label, h in result['shapes'].items():
        area = h.Integral('width' if ETA_NORMALIZATION == 'bin_width' else '')
        assert math.isclose(area, 1., abs_tol=1e-10), (label, interval, area)
    for model, ratio in result['ratios'].items():
        d, t = result['shapes']['Data'], result['shapes'][model]
        for i in ratio._valid_ratio_bins:
            D,T = d.GetBinContent(i),t.GetBinContent(i)
            error = math.hypot(d.GetBinError(i)/T, D*t.GetBinError(i)/T**2)
            assert math.isclose(ratio.GetBinContent(i), D/T, rel_tol=1e-10, abs_tol=1e-12)
            assert math.isclose(ratio.GetBinError(i), error, rel_tol=1e-10, abs_tol=1e-12)
    VALIDATION.append({'trigger': trigger, 'ptave': interval, 'bins': d.GetNbinsX(), 'status': 'passed'})
print(VALIDATION)
NOTEBOOK_ROOT_FILE = ROOT_OUTPUT.finish(globals())
print(NOTEBOOK_ROOT_FILE)